# Scope and Audit


In [1]:
import sys
from pathlib import Path

# Find the project root by walking up until src/olist is visible, so the
# notebook runs from any working directory.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "olist").exists())
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd

from olist import config as cfg, io_utils as io

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 220)

print("project root :", ROOT)
print("raw data dir :", cfg.RAW_DIR)
print("snapshot date:", cfg.SNAPSHOT_DATE.date(), "(fixed - never pd.Timestamp.now())")

project root : c:\Users\programmer\Desktop\program\ali-ajvand-data-portfolio\olist-delivery-analytics
raw data dir : C:\Users\programmer\Desktop\program\ali-ajvand-data-portfolio\olist-delivery-analytics\data\raw
snapshot date: 2018-10-18 (fixed - never pd.Timestamp.now())


## 1. Data provenance


In [2]:
provenance = {
    "dataset": "Olist Brazilian E-Commerce Public Dataset",
    "source": cfg.DATASET_SOURCE,
    "licence": "CC BY-NC-SA 4.0",
    "version": "single Kaggle release, 9 CSV files",
    "coverage": "Sep 2016 - Oct 2018, ~99k orders, ~3k sellers, 27 Brazilian states",
    "retrieved": "fill in the date you downloaded it",
    "cutoff_used": str(cfg.SNAPSHOT_DATE.date()),
    "known_caveats": [
        "revenue is a GMV proxy (item price); no cost of goods, no commission",
        "the delivery estimate is Olist's own and is deliberately generous",
        "geolocation is zip-prefix level, so distances are approximate",
        "orders can carry more than one review; reviews are deduplicated in phase 1",
    ],
}
for key, value in provenance.items():
    print(f"{key:>15}: {value}")

        dataset: Olist Brazilian E-Commerce Public Dataset
         source: Olist Brazilian E-Commerce Public Dataset (CC BY-NC-SA 4.0), https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce
        licence: CC BY-NC-SA 4.0
        version: single Kaggle release, 9 CSV files
       coverage: Sep 2016 - Oct 2018, ~99k orders, ~3k sellers, 27 Brazilian states
      retrieved: fill in the date you downloaded it
    cutoff_used: 2018-10-18
  known_caveats: ['revenue is a GMV proxy (item price); no cost of goods, no commission', "the delivery estimate is Olist's own and is deliberately generous", 'geolocation is zip-prefix level, so distances are approximate', 'orders can carry more than one review; reviews are deduplicated in phase 1']


## 2. Load raw data


In [3]:
raw = io.load_raw()
print(f"loaded {len(raw)} tables\n")
summary = io.table_summary(raw, cfg.EXPECTED_RAW_ROWS)
summary

loaded 9 tables



,table,rows,columns,memory_mb,null_cells_pct,duplicate_rows,expected_rows,rows_match
0,olist_customers_dataset,99441,5,26.6,0.00,0,99441,True
1,olist_geolocation_dataset,1000163,5,129.4,0.00,261831,1000163,True
2,olist_order_items_dataset,112650,7,36.0,0.00,0,112650,True
3,olist_order_payments_dataset,103886,5,16.2,0.00,0,103886,True
4,olist_order_reviews_dataset,99224,7,39.1,21.01,0,99224,True
5,olist_orders_dataset,99441,8,52.9,0.62,0,99441,True
6,olist_products_dataset,32951,9,6.3,0.83,0,32951,True
7,olist_sellers_dataset,3095,4,0.6,0.00,0,3095,True
8,product_category_name_translation,71,2,0.0,0.00,0,71,True


In [4]:
# Soft integrity gate. If this fails you have a re-sampled or truncated download,
# and every number downstream will quietly disagree with the published dataset.
if summary["expected_rows"].notna().all():
    mismatched = summary.loc[~summary["rows_match"].fillna(False), ["table", "rows", "expected_rows"]]
    if len(mismatched):
        print("ROW COUNTS DIFFER FROM THE CANONICAL RELEASE:")
        print(mismatched.to_string(index=False))
        print("\nThis is expected if you are running on scripts/make_sample_data.py output.")
    else:
        print("PASS - all nine tables match the canonical Kaggle row counts.")

PASS - all nine tables match the canonical Kaggle row counts.


## 3. Audit helper


In [5]:
def audit_table(df: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame(
        {
            "dtype": df.dtypes.astype(str),
            "non_null": df.notna().sum(),
            "null_pct": (df.isna().mean() * 100).round(2),
            "n_unique": df.nunique(),
            "example": [df[c].dropna().iloc[0] if df[c].notna().any() else None for c in df.columns],
        }
    )


for name, df in raw.items():
    print("=" * 100)
    print(f"{name}   rows={len(df):,}  cols={df.shape[1]}  duplicate_rows={df.duplicated().sum():,}")
    print(audit_table(df).to_string())

olist_customers_dataset   rows=99,441  cols=5  duplicate_rows=0
                           dtype  non_null  null_pct  n_unique                           example
customer_id               object     99441       0.0     99441  06b8999e2fba1a1fbc88172c00ba8bc7
customer_unique_id        object     99441       0.0     96096  861eff4711a542e4b93843c6dd7febb0
customer_zip_code_prefix   int64     99441       0.0     14994                             14409
customer_city             object     99441       0.0      4119                            franca
customer_state            object     99441       0.0        27                                SP
olist_geolocation_dataset   rows=1,000,163  cols=5  duplicate_rows=261,831
                               dtype  non_null  null_pct  n_unique    example
geolocation_zip_code_prefix    int64   1000163       0.0     19015       1037
geolocation_lat              float64   1000163       0.0    717360 -23.545621
geolocation_lng              float64   100016

## 4. Grain checks


In [6]:
grain_checks = [
    ("olist_orders_dataset", "order_id", "one row per order"),
    ("olist_customers_dataset", "customer_id", "one row per order-scoped customer"),
    ("olist_products_dataset", "product_id", "one row per product"),
    ("olist_sellers_dataset", "seller_id", "one row per seller"),
    ("olist_order_reviews_dataset", "order_id", "one review per order (EXPECTED TO FAIL)"),
    ("olist_geolocation_dataset", "geolocation_zip_code_prefix", "one point per zip (EXPECTED TO FAIL)"),
]

rows = []
for table, key, note in grain_checks:
    df = raw[table]
    rows.append(
        {
            "table": table,
            "key": key,
            "rows": len(df),
            "unique_keys": df[key].nunique(),
            "extra_rows": len(df) - df[key].nunique(),
            "unique": len(df) == df[key].nunique(),
            "note": note,
        }
    )
pd.DataFrame(rows)

,table,key,rows,unique_keys,extra_rows,unique,note
0,olist_orders_dataset,order_id,99441,99441,0,True,one row per order
1,olist_customers_dataset,customer_id,99441,99441,0,True,one row per order-scoped customer
2,olist_products_dataset,product_id,32951,32951,0,True,one row per product
3,olist_sellers_dataset,seller_id,3095,3095,0,True,one row per seller
4,olist_order_reviews_dataset,order_id,99224,98673,551,False,one review per order (EXPECTED TO FAIL)
5,olist_geolocation_dataset,geolocation_zip_code_prefix,1000163,19015,981148,False,one point per zip (EXPECTED TO FAIL)


## 5. Orders audit


In [7]:
orders = raw["olist_orders_dataset"].copy()
for col in ("order_purchase_timestamp", "order_delivered_customer_date", "order_estimated_delivery_date"):
    orders[col] = pd.to_datetime(orders[col], errors="coerce")

print("purchase window :", orders["order_purchase_timestamp"].min(), "->", orders["order_purchase_timestamp"].max())
print("delivery window :", orders["order_delivered_customer_date"].min(), "->", orders["order_delivered_customer_date"].max())
print()
print("order status mix:")
print(orders["order_status"].value_counts(dropna=False).to_frame("orders").assign(
    share_pct=lambda d: (d["orders"] / len(orders) * 100).round(2)
))
print()
undelivered = orders["order_delivered_customer_date"].isna()
print(f"orders with no delivery date: {undelivered.sum():,} ({undelivered.mean():.2%})")
print("These are NOT on-time deliveries. Their target is NaN, never 0.")

purchase window : 2016-09-04 21:15:19 -> 2018-10-17 17:30:18
delivery window : 2016-10-11 13:46:32 -> 2018-10-17 13:22:46

order status mix:
              orders  share_pct
order_status                   
delivered      96478      97.02
shipped         1107       1.11
canceled         625       0.63
unavailable      609       0.61
invoiced         314       0.32
processing       301       0.30
created            5       0.01
approved           2       0.00

orders with no delivery date: 2,965 (2.98%)
These are NOT on-time deliveries. Their target is NaN, never 0.


In [8]:
monthly = (
    orders.assign(month=orders["order_purchase_timestamp"].dt.to_period("M"))
    .groupby("month")
    .agg(orders=("order_id", "nunique"), delivered=("order_delivered_customer_date", "count"))
)
monthly["delivered_pct"] = (monthly["delivered"] / monthly["orders"] * 100).round(1)
monthly.tail(8)

,orders,delivered,delivered_pct
month,,,
2018-03,7211,7003,97.1
2018-04,6939,6798,98.0
2018-05,6873,6749,98.2
2018-06,6167,6096,98.8
2018-07,6292,6156,97.8
2018-08,6512,6351,97.5
2018-09,16,0,0.0
2018-10,4,0,0.0


## 6. Summary
